# Topic 5: Lakehouse Architecture & Governance

> **Phase 7 → Week 13 → Topic 5**

---

## What Is a Lakehouse?

A Lakehouse combines the scalability and cost-effectiveness of a data lake (S3/GCS/ADLS) with the ACID transactions, schema enforcement, and BI performance of a data warehouse.

```
Data Lake:        Cheap storage, any format, no ACID → analysts can't trust data
Data Warehouse:   ACID, fast SQL, expensive, can't store unstructured data
Lakehouse:        ACID on cheap object storage (Delta/Iceberg/Hudi) + fast SQL engines
```

---

## Interview Cheat Sheet

**Q: What is the Lakehouse architecture and which open formats implement it?**
> A Lakehouse is an open data architecture that adds a metadata/transaction layer on top of cloud object storage. This enables ACID transactions, schema enforcement, time travel, and efficient upserts on data that lives in cheap S3/GCS/ADLS. Three open formats: Delta Lake (Databricks, Apache), Apache Iceberg (Netflix, Apple, now broad support), Apache Hudi (Uber, AWS). All three are now supported by major query engines (Spark, Flink, Trino, Athena, Snowflake).

**Q: What is data mesh and how does it differ from a centralized data lake?**
> Data mesh is a decentralized architecture where domain teams own their own data products end-to-end (ingestion → quality → serving). A centralized data lake has a single data team owning everything — this becomes a bottleneck. In data mesh: the payments team owns and publishes the `payments` domain data product with its own SLAs and quality guarantees. A central platform team provides infrastructure (catalog, storage, pipeline framework), but not data itself. Tradeoff: data mesh improves autonomy and quality but requires engineering maturity in every domain team.

**Q: What is Apache Iceberg and how does it compare to Delta Lake?**
> Apache Iceberg is an open table format for large analytic tables. Like Delta Lake, it provides ACID transactions, schema evolution, time travel, and hidden partitioning. Key differences: Iceberg uses a tree of metadata files (manifest lists → manifests → data files) rather than a transaction log; Iceberg's hidden partitioning auto-partitions without changing queries (no `WHERE date=...` needed); Iceberg has broader multi-engine support (Spark + Flink + Trino + Dremio + Snowflake). Delta Lake is tighter with Databricks and has better auto-optimize features (Auto Loader, OPTIMIZE, ZORDER). AWS Athena natively supports both.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os, shutil

spark = SparkSession.builder \
    .appName("Week13 - Lakehouse Architecture") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
BASE = "/tmp/lakehouse_demo"
if os.path.exists(BASE): shutil.rmtree(BASE)
os.makedirs(BASE)
print("Lakehouse Architecture — ready")

## Part 1: Medallion Architecture Deep Dive

In [ ]:
from delta.tables import DeltaTable

BRONZE = f"{BASE}/bronze/orders"
SILVER = f"{BASE}/silver/orders"
GOLD   = f"{BASE}/gold/customer_revenue"

# ── Bronze: raw, as-is ingest ────────────────────────────────
# Bronze keeps everything — no filtering, no transforms
# Adds: ingestion timestamp, source file, job run ID
raw_orders = spark.createDataFrame([
    ("ORD001", "C001", 250.0,  "shipped",   "2024-01-15 10:00"),
    ("ORD002", None,  -50.0,  "pending",    "2024-01-15 10:01"),  # invalid: no customer
    ("ORD003", "C002", 180.0,  "cancelled", "2024-01-15 10:02"),
    ("ORD001", "C001", 250.0,  "shipped",   "2024-01-15 10:00"),  # duplicate
], ["order_id", "customer_id", "amount", "status", "event_time"])

bronze = raw_orders \
    .withColumn("_ingested_at", F.current_timestamp()) \
    .withColumn("_source", F.lit("orders_api")) \
    .withColumn("_run_id", F.lit("run_20240115_001"))

bronze.write.format("delta").mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .save(BRONZE)

print("Bronze layer (raw, unfiltered):")
spark.read.format("delta").load(BRONZE).show(truncate=False)

# ── Silver: validated, deduplicated, typed ────────────────────
# Silver applies business rules — rejects/quarantines bad records
bronze_df = spark.read.format("delta").load(BRONZE)

valid_orders = bronze_df.filter(
    F.col("customer_id").isNotNull() &
    (F.col("amount") > 0)
).dropDuplicates(["order_id"]) \
 .withColumn("event_ts", F.to_timestamp("event_time")) \
 .select("order_id", "customer_id", "amount", "status", "event_ts", "_ingested_at")

# MERGE: idempotent silver layer
valid_orders.write.format("delta").mode("overwrite").save(SILVER)

print("\nSilver layer (validated, deduped):")
spark.read.format("delta").load(SILVER).show(truncate=False)

# ── Gold: aggregated, business-ready ─────────────────────────
silver_df = spark.read.format("delta").load(SILVER)

gold_df = silver_df \
    .filter(F.col("status") == "shipped") \
    .groupBy("customer_id", F.to_date("event_ts").alias("date")) \
    .agg(
        F.sum("amount").alias("revenue"),
        F.count("*").alias("order_count"),
        F.avg("amount").alias("avg_order_value")
    )

gold_df.write.format("delta").mode("overwrite").save(GOLD)
print("\nGold layer (business aggregations):")
spark.read.format("delta").load(GOLD).show()

## Part 2: Open Table Format Comparison

In [ ]:
print("""
Open Table Formats — Delta vs Iceberg vs Hudi:
══════════════════════════════════════════════════════════════

Feature                 Delta Lake          Apache Iceberg       Apache Hudi
─────────────────────  ─────────────────   ──────────────────   ──────────────────
ACID transactions      ✓                   ✓                    ✓
Time travel            ✓ (versions)        ✓ (snapshots)        ✓ (commit timeline)
Schema evolution       ✓                   ✓ (hidden partition) ✓
UPSERT/MERGE           ✓                   ✓                    ✓ (native upsert)
File format            Parquet             Parquet/ORC/Avro     Parquet/ORC
Partitioning           Explicit            Hidden (auto)        Explicit
Streaming ingest       ✓ (Spark SS)        ✓ (Flink native)     ✓ (Hudi native)
Spark support          Best                Good                 Good
Flink support          Limited             Excellent            Good
Trino/Presto           Good                Excellent            Good
Athena support         ✓ (v3)              ✓ (native)           ✓ (via manifest)
Snowflake support      External tables     ✓                    External tables
Best ecosystem fit     Databricks + S3     Multi-engine         AWS + streaming
Creator                Databricks          Netflix/Apple        Uber

Iceberg Hidden Partitioning (killer feature):
  Delta: you partition by date → query MUST use WHERE date='2024-01-15'
  Iceberg: you declare partition transforms → query uses WHERE ts='...'
           Iceberg automatically maps timestamps → partitions
  Result: changing partition strategy doesn't break existing queries

When to choose:
  Delta Lake:   Databricks shop, want best Spark/Python integration
  Iceberg:      Multi-engine (Spark + Flink + Trino), avoid vendor lock-in
  Hudi:         AWS-native, need high-velocity upserts (CDC from databases)
══════════════════════════════════════════════════════════════
""")

## Part 3: Data Governance Patterns

In [ ]:
print("""
Data Governance in a Lakehouse:
══════════════════════════════════════════════════════════════

1. Data Catalog (discoverability):
   AWS: Glue Data Catalog + DataZone
   Databricks: Unity Catalog
   Open-source: Apache Atlas, DataHub (LinkedIn), OpenMetadata
   → Every table/column has owner, description, tags, lineage

2. Access Control (who can read/write what):
   Coarse: IAM bucket policies (read this S3 path)
   Fine-grained: column-level (mask PII), row-level (region filter)
   Databricks: Unity Catalog GRANT/REVOKE
   AWS Lake Formation: centralized ACL for Glue Catalog tables

3. Data Lineage (where did this data come from?):
   Automatic: Databricks Unity Catalog, Apache Atlas
   Manual: dbt lineage graph (SQL model dependencies)
   OpenLineage: open protocol for lineage (supported by Airflow, Spark)

4. Data Quality (is the data trustworthy?):
   At ingest (Silver): DLT expectations, Great Expectations, custom checks
   Ongoing: dbt tests, Soda Core, Monte Carlo (anomaly detection)
   SLAs: max null rate, row count bounds, freshness check

5. PII/Compliance:
   Discovery: AWS Macie (auto-detects PII in S3)
   Masking: column masks in Unity Catalog / Lake Formation
   Deletion: GDPR right-to-erasure → Delta MERGE to NULL or DELETE
   Audit: CloudTrail + Glue Catalog access logs

AWS Lake Formation (governance layer over Glue Catalog):
  - Register S3 locations (Lake Formation manages access)
  - Grant column-level and row-level permissions
  - Works with: Athena, EMR, Glue, Redshift Spectrum
  - Replaces complex IAM bucket policies with a table-level API

  Example:
  GRANT SELECT(order_id, customer_id, status) -- not amount
  ON TABLE analytics.orders
  TO ROLE analyst_group;
══════════════════════════════════════════════════════════════
""")

## Part 4: Data Mesh Concepts

In [ ]:
print("""
Data Mesh — Four Principles:
══════════════════════════════════════════════════════════════

1. Domain-oriented ownership
   Bad:  Central data team owns ALL data → bottleneck → stale data
   Good: Payments team owns payments data end-to-end
         Orders team owns orders data end-to-end
         Each domain: ingest → quality → publish as a data product

2. Data as a product
   Each domain publishes a data product with:
   - Discoverable schema (in catalog)
   - Documented SLAs (freshness, availability, quality)
   - Versioned API (schema changes are backwards-compatible)
   - Owner accountability (who to call when it breaks)

3. Self-serve data platform
   Central platform team provides infrastructure:
   - Storage (S3 + Delta/Iceberg)
   - Compute (EMR / Databricks cluster policies)
   - Catalog + Governance (Glue Catalog / Unity Catalog)
   - CI/CD pipeline templates
   Domain teams consume platform, don't manage infra

4. Federated computational governance
   Global policies enforced automatically:
   - PII columns must be masked for external consumers
   - All tables must have an owner and description
   - No data product with >5% null rate on key columns
   Domain teams apply policies through config, not custom code

Data Mesh vs Centralized Lake:
  Centralized: one team, fast consistency, bottleneck at scale
  Data Mesh: autonomous teams, scales with org, high coordination cost
  Choose Data Mesh when: 50+ engineers, multiple product domains,
                         central team is the bottleneck

Data Mesh implementation on AWS:
  Storage:     S3 per domain (orders-data-prod, payments-data-prod)
  Catalog:     Glue Catalog + AWS DataZone (cross-account discovery)
  Governance:  Lake Formation (cross-account table sharing)
  Compute:     Each domain has their own EMR/Glue resources
  Sharing:     Lake Formation RAM shares → domain A grants domain B access
══════════════════════════════════════════════════════════════
""")

## Exercises

1. Implement a complete Medallion Architecture using Delta Lake for an e-commerce platform. Bronze ingests raw order JSON (keep everything). Silver validates (non-null customer, positive amount, valid status), deduplicates on order_id, and casts types. Gold builds daily revenue and top-5 products per category. Make each layer idempotent.
2. Compare Delta Lake and Apache Iceberg on three criteria important to a team that uses Spark + Flink + Trino. Which would you recommend and why?
3. A user exercises their GDPR right-to-erasure. You have their data in Bronze (raw) and Silver (validated) Delta tables. Write a plan and Delta SQL to delete all their rows. What are the complications with Bronze (which is append-only by design)?
4. Design a data mesh for a company with three domains: Orders, Payments, Inventory. For each domain: what data products do they publish, what SLAs make sense, and what governance rules should be globally enforced?
5. What are the tradeoffs of using partition pruning in Delta Lake (`WHERE date='2024-01-15'`) vs Iceberg's hidden partitioning? If you frequently change how data is partitioned (e.g., daily → hourly), which format is safer to evolve?